# 3.5 Functions & Lambda Functions

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 3: Python for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-03-python-for-machine-learning/03.05_functions-and-lambda-functions.ipynb)

Companion code for Section 3.5 of the book. Run the cells in order from top to bottom.

**Example 3.5.1**

In [ ]:
# Basic function definition
def calculate_accuracy(y_true, y_pred):
    '''
    Calculate classification accuracy.

    Args:
        y_true: list or array of true labels
        y_pred: list or array of predicted labels
    Returns:
        float: accuracy in [0, 1]
    '''
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    return correct / len(y_true)

y_true = [1, 0, 1, 1, 0, 1, 0, 0]
y_pred = [1, 0, 0, 1, 0, 1, 1, 0]
acc = calculate_accuracy(y_true, y_pred)
print(f'Accuracy: {acc:.2%}')   # 75.00%

# Default arguments
def train_model(X, y, lr=0.01, epochs=100, verbose=True):
    '''Train with sensible defaults - user only specifies what they want to change.'''
    losses = []
    import random; random.seed(42)
    loss = 1.0
    for ep in range(epochs):
        loss -= lr * random.uniform(0.5, 1.5)
        loss = max(0.01, loss)
        losses.append(loss)
        if verbose and (ep+1) % 20 == 0:
            print(f'  Epoch {ep+1:3d}: loss={loss:.4f}')
    return losses

losses = train_model(None, None, lr=0.05, epochs=60)  # Override defaults
print(f'Final loss: {losses[-1]:.4f}')

**Example 3.5.2**

In [ ]:
# *args: accept any number of positional arguments as a tuple
def log_metrics(*metrics):
    for name, value in metrics:
        print(f'  {name}: {value:.4f}')

log_metrics(('accuracy', 0.94), ('loss', 0.0421), ('f1', 0.935))

# **kwargs: accept any number of keyword arguments as a dict
def build_config(**kwargs):
    defaults = {'lr': 0.001, 'batch': 32, 'epochs': 100, 'optimizer': 'adam'}
    defaults.update(kwargs)   # Override defaults with provided kwargs
    return defaults

config = build_config(lr=0.0001, epochs=50, dropout=0.3)
print(config)

# Combining *args and **kwargs - maximum flexibility
def experiment(model_name, *datasets, **hyperparams):
    print(f'Model: {model_name}')
    print(f'Datasets: {datasets}')
    print(f'Hyperparams: {hyperparams}')

experiment('ResNet50', 'ImageNet', 'CIFAR10', lr=0.001, epochs=200, gpu=True)

# Unpacking with * and ** when calling functions
def compute_metrics(accuracy, precision, recall):
    return (2 * precision * recall) / (precision + recall)  # F1

scores = [0.94, 0.91, 0.96]
metrics_dict = {'accuracy': 0.94, 'precision': 0.91, 'recall': 0.96}

f1 = compute_metrics(*scores)              # Unpack list as positional args
f1 = compute_metrics(**metrics_dict)       # Unpack dict as keyword args
print(f'F1 score: {f1:.4f}')               # 0.9347

**Example 3.5.3**

In [ ]:
# Basic lambda syntax: lambda arguments: expression
square  = lambda x: x**2
sigmoid = lambda x: 1 / (1 + 2.718**(-x))
relu    = lambda x: max(0, x)

print(square(5))       # 25
print(sigmoid(0))      # ~0.5
print(relu(-3))        # 0

# Lambda most useful as argument to higher-order functions
models = [
    {'name': 'LogReg',   'accuracy': 0.87, 'train_time': 0.5},
    {'name': 'XGBoost',  'accuracy': 0.94, 'train_time': 12.3},
    {'name': 'ResNet',   'accuracy': 0.97, 'train_time': 450.0},
    {'name': 'LightGBM', 'accuracy': 0.93, 'train_time': 5.8},
]

# Sort by accuracy descending
by_accuracy = sorted(models, key=lambda m: m['accuracy'], reverse=True)
print('By accuracy:')
for m in by_accuracy:
    print(f"  {m['name']:12s}: {m['accuracy']:.2f}")

# Filter: only fast models (< 60 seconds)
fast_models = list(filter(lambda m: m['train_time'] < 60, models))
print(f'\nFast models: {[m["name"] for m in fast_models]}')

# Map: convert train_time to minutes
in_minutes = list(map(lambda m: {**m, 'train_time_min': m['train_time']/60}, models))

# min/max with key function
best  = max(models, key=lambda m: m['accuracy'])
fastest = min(models, key=lambda m: m['train_time'])
print(f'Best:    {best["name"]} ({best["accuracy"]:.2f})')
print(f'Fastest: {fastest["name"]} ({fastest["train_time"]}s)')

**Example 3.5.4**

In [ ]:
import time
import functools

# Closure: make_multiplier returns a function that 'remembers' factor
def make_scaler(factor):
    def scale(x):               # 'factor' is captured from enclosing scope
        return x * factor
    return scale

normalise = make_scaler(1/255)  # Scale pixel values to [0,1]
print(normalise(128))           # 0.5019...

# Decorator: a function that wraps another function
def timer(func):
    @functools.wraps(func)      # Preserves original function's metadata
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f'[timer] {func.__name__} took {elapsed*1000:.2f}ms')
        return result
    return wrapper

def validate_input(func):
    @functools.wraps(func)
    def wrapper(X, y, **kwargs):
        if len(X) != len(y):
            raise ValueError(f'X ({len(X)}) and y ({len(y)}) must have same length')
        if len(X) == 0:
            raise ValueError('Empty dataset provided')
        print(f'[validate] Input OK: {len(X)} samples')
        return func(X, y, **kwargs)
    return wrapper

@timer             # Apply timer decorator
@validate_input    # Apply validation decorator (applied first, bottom-up)
def train(X, y, epochs=10):
    '''Simulated training function.'''
    import random; random.seed(0)
    return [1.0 - i*0.08 + random.uniform(-0.01, 0.01) for i in range(epochs)]

X = list(range(100)); y = list(range(100))
history = train(X, y, epochs=5)
print('Loss history:', [f'{l:.3f}' for l in history])